# LPNE2GGCN — End-to-End Demo

This notebook walks through the full LPNE2GGCN pipeline step by step:

1. Load a dataset (Cora)
2. Construct positive/negative edge pairs (Algorithm 1)
3. Learn Node2Vec embeddings
4. Refine with GraphSAGE
5. Evaluate (Accuracy, AUC, F1)
6. Reproduce paper figures


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import torch
import matplotlib.pyplot as plt

from utils.data_loader        import load_dataset
from utils.negative_sampling  import prepare_link_prediction_data
from utils.metrics            import compute_metrics
from utils.visualize          import plot_accuracy_comparison, plot_auc_comparison, plot_timing_chart
from models.lpne2ggcn         import LPNE2GGCN

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Load Dataset

In [ ]:
graph = load_dataset('cora')
print(f"Nodes   : {graph['num_nodes']}")
print(f"Edges   : {graph['num_edges']}")
print(f"Features: {graph['num_feats']}")

## 2. Build Link Prediction Dataset (Algorithm 1)

In [ ]:
lp_data = prepare_link_prediction_data(
    graph,
    test_ratio=0.2,
    pos_ratio=0.5,
    max_neg_distance=3,
    seed=SEED,
)
print(f"Train pairs : {len(lp_data['train_edges'])}")
print(f"Test  pairs : {len(lp_data['test_edges'])}")
print(f"Positive (train): {sum(lp_data['train_labels'])}")
print(f"Negative (train): {sum(1-l for l in lp_data['train_labels'])}")

## 3. Train LPNE2GGCN (Adam optimizer)

In [ ]:
model = LPNE2GGCN(
    embedding_dim=64,
    hidden_dim=128,
    num_layers=3,      # Table 6: 3-layers is optimal
    aggregator='mean',
    dropout=0.3,
    p=1.0, q=1.0,      # Section IV-D optimal parameters
    walk_length=10,
    num_walks=10,
    optimizer='adam',
    epochs=200,
    seed=SEED,
)

model.fit(
    G_train       = lp_data['G_train'],
    edge_index    = lp_data['edge_index_train'],
    train_edges   = lp_data['train_edges'],
    train_labels  = lp_data['train_labels'],
    node_features = graph['x'].numpy(),
)

## 4. Evaluate

In [ ]:
metrics = model.evaluate(lp_data['test_edges'], lp_data['test_labels'])

print(f"Accuracy : {metrics['accuracy']:.2f}%  (paper: 96.46%)")
print(f"AUC      : {metrics['auc']:.2f}%  (paper: 97.46%)")
print(f"F1-Score : {metrics['f1']:.2f}%  (paper: 96.46%)")

## 5. Reproduce Paper Figures

In [ ]:
plot_accuracy_comparison(save=False)

In [ ]:
plot_auc_comparison(save=False)

In [ ]:
plot_timing_chart(save=False)

## 6. Layer Ablation (Table 6)

The paper shows that 3-layer GraphSAGE is optimal across all datasets.

In [ ]:
from utils.visualize import plot_layer_ablation
plot_layer_ablation(save=False)